# Financial RAG Chatbot with LangSmith Evaluation

In [1]:
import os
import json
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.chat_models import ChatOllama
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langsmith import Client
from langsmith.schemas import Example, Run
import pandas as pd
from datetime import datetime

# Initialize LangSmith client (requires LANGSMITH_API_KEY and LANGSMITH_ENDPOINT env vars)
client = Client()
print("✓ LangSmith client initialized")

c:\Users\ansul\OneDrive\Desktop\data science project\financial_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'langchain.memory'

## 1. Load Vector Store and Initialize Embeddings

In [ ]:
# Initialize embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/e5-large",
    model_kwargs={"device": "cpu"},  # Change to "cuda" if GPU available
    encode_kwargs={"normalize_embeddings": True}
)
print("✓ Embeddings model loaded (e5-large)")

# Load FAISS vector store
vectorstore_path = r"..\data\embeddings_E5_Large"
vectorstore = FAISS.load_local(
    vectorstore_path,
    embedding_model,
    allow_dangerous_deserialization=True
)
print(f"✓ Vector store loaded from {vectorstore_path}")
print(f"  Database size: {vectorstore.index.ntotal} vectors")

## 2. Create Retriever with Metadata Support

In [ ]:
def format_docs(docs):
    """Format retrieved documents for the prompt."""
    formatted = []
    for i, doc in enumerate(docs, 1):
        metadata = doc.metadata
        formatted.append(
            f"[Source {i}] {metadata.get('company_name', 'Unknown')} - "
            f"{metadata.get('section', 'Unknown').replace('_', ' ').title()}\n{doc.page_content}\n"
        )
    return "\n---\n".join(formatted)

# Create retriever with k=5 as default
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)
print("✓ Retriever initialized (k=5)")

## 3. Setup LLM

In [ ]:
# Option 1: Use OpenAI (requires OPENAI_API_KEY)
try:
    llm = ChatOpenAI(
        model_name="gpt-3.5-turbo",
        temperature=0.3,  # Lower temp for factual answers
        max_tokens=1024
    )
    print("✓ Using OpenAI GPT-3.5-turbo")
except Exception as e:
    print(f"⚠ OpenAI not available ({e})")
    # Option 2: Use Ollama locally (requires ollama running locally)
    try:
        llm = ChatOllama(
            model="mistral",  # or llama2, neural-chat, etc.
            temperature=0.3,
            base_url="http://localhost:11434"
        )
        print("✓ Using Ollama (Mistral)")
    except Exception as e:
        print(f"⚠ Ollama not available either ({e})")
        raise Exception("Please set OPENAI_API_KEY or run Ollama locally")

## 4. Create RAG Chain with System Prompt

In [ ]:
# Create RAG prompt
rag_prompt = ChatPromptTemplate.from_template("""You are an expert financial analyst assistant specializing in 10-K filing analysis.
You have access to financial documents from various companies. Use the provided context to answer questions accurately.

Guidelines:
- Answer only based on the provided documents
- If information is not available in the context, clearly state that
- Cite the source company and section when providing specific information
- Be precise with numbers, percentages, and financial metrics
- For multi-company comparisons, make it clear which company each point refers to

Context from financial documents:
{context}

Question: {question}

Answer:"""
)

# Create the RAG chain
rag_chain = (
    {"context": retriever | RunnablePassthrough.map(format_docs), "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✓ RAG chain created and ready for queries")

## 5. Create Chat Manager with Memory and Evaluation Hooks

In [ ]:
class FinancialChatBot:
    """Financial RAG chatbot with memory and LangSmith evaluation tracking."""
    
    def __init__(self, rag_chain, client=None, project_name="financial-rag-chat"):
        self.rag_chain = rag_chain
        self.client = client
        self.project_name = project_name
        self.chat_history = []
        self.conversation_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        
    def chat(self, user_question: str, log_to_langsmith=True) -> dict:
        """Send a question and get response with optional LangSmith logging."""
        
        # Get response from RAG chain
        response = self.rag_chain.invoke(user_question)
        
        # Store in history
        self.chat_history.append({
            "timestamp": datetime.now().isoformat(),
            "user_question": user_question,
            "response": response
        })
        
        # Log to LangSmith if available
        if log_to_langsmith and self.client:
            try:
                self.client.create_run(
                    name="financial_rag_query",
                    inputs={"question": user_question},
                    outputs={"answer": response},
                    project_name=self.project_name,
                    tags=["rag", "financial"]
                )
            except Exception as e:
                print(f"⚠ LangSmith logging failed: {e}")
        
        return {
            "question": user_question,
            "answer": response,
            "timestamp": datetime.now().isoformat()
        }
    
    def get_history(self):
        """Return chat history as dataframe for analysis."""
        return pd.DataFrame(self.chat_history)
    
    def save_session(self, filepath: str):
        """Save chat session to JSON."""
        session_data = {
            "conversation_id": self.conversation_id,
            "timestamp": datetime.now().isoformat(),
            "chat_history": self.chat_history
        }
        with open(filepath, "w") as f:
            json.dump(session_data, f, indent=2)
        print(f"✓ Session saved to {filepath}")

# Initialize chatbot
chatbot = FinancialChatBot(rag_chain, client=client)
print("✓ FinancialChatBot initialized")

## 6. LangSmith Evaluation Setup

In [ ]:
# Define evaluation functions
def evaluate_relevance(run: Run, example: Example) -> dict:
    """Check if response contains relevant information from retrieved documents."""
    prediction = run.outputs.get("answer", "")
    
    # Simple heuristic: check for specific financial metrics/terms
    financial_indicators = ["million", "billion", "percent", "$", "revenue", "risk", "operating", "net income"]
    relevance_score = sum(1 for term in financial_indicators if term.lower() in prediction.lower())
    
    return {"relevance_score": min(relevance_score / 3, 1.0)}  # Normalized to 0-1

def evaluate_accuracy(run: Run, example: Example) -> dict:
    """Check if response cites sources and avoids hallucinations."""
    prediction = run.outputs.get("answer", "")
    
    # Check for hallucination indicators
    hallucination_indicators = ["i don't have access", "i cannot provide", "no information"]
    is_honest = any(term in prediction.lower() for term in hallucination_indicators)
    
    # Check for source attribution
    has_sources = "[source" in prediction.lower() or "according to" in prediction.lower()
    
    accuracy_score = 0.5 if is_honest else 0.7
    accuracy_score += 0.3 if has_sources else 0
    
    return {"accuracy_score": accuracy_score}

def evaluate_length(run: Run, example: Example) -> dict:
    """Check if response length is appropriate."""
    prediction = run.outputs.get("answer", "")
    word_count = len(prediction.split())
    
    # Ideal range: 100-500 words for financial questions
    if 100 <= word_count <= 500:
        length_score = 1.0
    elif word_count < 50:
        length_score = 0.3
    elif word_count > 1000:
        length_score = 0.5
    else:
        length_score = 0.7
    
    return {"length_score": length_score, "word_count": word_count}

print("✓ Evaluation functions defined")

## 7. Example Chat Interactions

In [ ]:
# Example queries to test the system
test_queries = [
    "What are the main risk factors mentioned in the 10-K filings?",
    "Compare the revenue trends across different companies.",
    "What cybersecurity risks are disclosed by the companies?",
    "Summarize the management discussion and analysis section.",
    "What are the key segments or business divisions mentioned?"
]

# Test with one query
print("Testing chatbot with example query...\n")
question = test_queries[0]
print(f"Q: {question}\n")

result = chatbot.chat(question, log_to_langsmith=True)
print(f"A: {result['answer']}\n")
print(f"Response time: {datetime.now().isoformat()}")

## 8. Interactive Chat Loop & Session Management

In [ ]:
def run_interactive_chat():
    """Run interactive chat loop in notebook."""
    print("=" * 70)
    print("FINANCIAL RAG CHATBOT - Interactive Mode")
    print("=" * 70)
    print("Type 'quit' or 'exit' to end conversation")
    print("Type 'save' to save the session")
    print("Type 'history' to view chat history")
    print("=" * 70 + "\n")
    
    while True:
        try:
            user_input = input("You: ").strip()
            
            if not user_input:
                continue
            
            if user_input.lower() in ['quit', 'exit']:
                print("\nGoodbye!")
                break
            
            if user_input.lower() == 'history':
                history_df = chatbot.get_history()
                print("\n--- Chat History ---")
                print(history_df.to_string())
                print()
                continue
            
            if user_input.lower() == 'save':
                session_file = f"session_{chatbot.conversation_id}.json"
                chatbot.save_session(session_file)
                continue
            
            # Get response
            result = chatbot.chat(user_input)
            print(f"\nAssistant: {result['answer']}\n")
            
        except KeyboardInterrupt:
            print("\n\nChat interrupted. Session data preserved.")
            break
        except Exception as e:
            print(f"\nError: {e}\n")

# Run interactive chat (uncomment to run)
# run_interactive_chat()

## 9. Batch Evaluation & Performance Analytics

In [ ]:
def run_batch_evaluation(queries: list, project_name: str = "financial-rag-eval"):
    """Run evaluation on multiple queries and return metrics."""
    
    results = []
    
    for i, query in enumerate(queries, 1):
        print(f"\n[{i}/{len(queries)}] Query: {query[:60]}...")
        
        # Get response
        result = chatbot.chat(query, log_to_langsmith=True)
        
        # Extract metrics
        evaluation = {
            "query_num": i,
            "query": query,
            "response": result["answer"][:100] + "..." if len(result["answer"]) > 100 else result["answer"],
            "response_length": len(result["answer"].split()),
        }
        
        results.append(evaluation)
        print(f"  ✓ Response length: {evaluation['response_length']} words")
    
    # Create results dataframe
    results_df = pd.DataFrame(results)
    
    # Calculate aggregate metrics
    print("\n" + "=" * 70)
    print("BATCH EVALUATION RESULTS")
    print("=" * 70)
    print(f"Total queries: {len(queries)}")
    print(f"Average response length: {results_df['response_length'].mean():.1f} words")
    print(f"Response length range: {results_df['response_length'].min()}-{results_df['response_length'].max()} words")
    print("=" * 70)
    
    return results_df

# Run batch evaluation on sample queries
print("Starting batch evaluation...")
eval_results = run_batch_evaluation(test_queries[:3])  # Test with first 3 queries

## Configuration & Setup Guide

### Prerequisites
1. **LangSmith Setup:**
   - Get API key from https://smith.langchain.com/
   - Set environment variables:
     ```
     LANGSMITH_API_KEY=<your-api-key>
     LANGSMITH_ENDPOINT=https://api.smith.langchain.com
     ```

2. **LLM Options:**
   - **OpenAI:** Set `OPENAI_API_KEY` environment variable
   - **Ollama:** Run `ollama serve` then `ollama pull mistral` (or other model)

3. **Dependencies:**
   ```bash
   pip install langchain langchain-openai langchain-community langsmith faiss-cpu sentence-transformers pandas
   ```

### Key Features
- ✓ Vector retrieval with metadata filtering
- ✓ Automatic chat history tracking
- ✓ LangSmith integration for evaluation
- ✓ Batch evaluation support
- ✓ Session persistence
- ✓ Multiple LLM backend support

### Next Steps
1. Comment/uncomment your preferred LLM in Section 3
2. Add your LangSmith API key to environment
3. Run cells sequentially to test
4. Use `run_batch_evaluation()` to evaluate system performance
5. Check LangSmith dashboard to view traces and evaluations